# 01 · Data cleaning

This notebook runs `src/cleaning.py`. It reads the Kaggle database, checks every table, and builds
one row per player and year in `data/cleaned_player_years.csv.gz`. Then it shows the main checks.

A row is a player who started at least one league match in the season that just ended. On 1 August
we know his latest rating. One year later we look at his latest rating again. If it went up by at
least the year's median change + 3 points, the row is a **breakout**.

In [ ]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").exists())
sys.path.insert(0, str(ROOT))
import pandas as pd
from src import cleaning
from src.common import CLEANED, DATA, TABLES
pd.set_option("display.width", 160, "display.max_columns", 20, "display.max_colwidth", 80)
cleaning.main()

## Rows per year

`paired_ratings` is the number of rows we can use: the player has a rating both before and after 1 August. 2015 is not complete because the data stops on 7 July 2016.

In [ ]:
pd.read_csv(DATA / "coverage.csv")

## Every check we ran

Each line is one check, the number we found, and what we did about it.

In [ ]:
pd.read_csv(DATA / "cleaning_audit.csv")

## Who we lose

Players with no rating a year later cannot be labelled, so they drop out. They are older, rated lower and play in smaller leagues, so our results describe players who stay in EA's database.

In [ ]:
print((DATA / "selection_bias.csv").read_text())

## Missing line-ups

The league-seasons with the most missing player IDs in the line-ups. Starts are undercounted there, so we keep a coverage column for each club.

In [ ]:
lineups = pd.read_csv(DATA / "lineup_coverage.csv")
lineups.sort_values("mean_missing_ids", ascending=False).head(10)

## One cleaned row

Mark Davies on 1 August 2011: rated 72 with potential 78, rated 75 a year later. The median change that year was 0, so +3 is a breakout.

In [ ]:
cleaned = pd.read_csv(CLEANED)
cleaned[(cleaned.player_name == "Mark Davies") & (cleaned.year == 2011)][
    ["year", "player_name", "age", "starts", "overall_rating", "potential", "potential_gap",
     "next_rating", "change", "annual_median_change", "breakout"]]

## Baseline

Ranking players by EA's potential minus overall rating, against a random list of the same size (10% of each year's players).

In [ ]:
pd.read_csv(TABLES / "baseline.csv").round(3)